# Batch Data Extraction at Scale with Cortex AI Functions

This notebook demonstrates how to build a batch data extraction pipeline using Snowflake Cortex AI Functions across text, images, video, and audio.

**Prerequisites:** Run `setup.sql` and `setup-post-upload.sql` before starting this notebook to create all required objects and load data.

## 1. Text Extraction with AI_EXTRACT

Extract structured fields from free-text customer reviews using AI_EXTRACT.

In [ ]:
%%sql -r dataframe_20
USE ROLE SYSADMIN;

In [ ]:
%%sql -r dataframe_1
-- Entity extraction from reviews
SELECT 
    REVIEW,
    AI_EXTRACT(
        text => REVIEW,
        responseFormat => {
            'truck_name': 'What food truck or brand is mentioned?',
            'dish': 'What specific dish or menu item is mentioned?',
            'issue_type': 'What type of issue did the customer experience (food quality, service, wait time, cleanliness, none)?',
            'would_recommend': 'Would the customer recommend this food truck (yes, no, unclear)?'
        }
    ) AS extracted_fields
FROM TB_VOC.ANALYTICS.TRUCK_REVIEWS_V_SAMPLE
LIMIT 10;

In [ ]:
%%sql -r dataframe_2
-- Extraction with confidence scores
SELECT 
    REVIEW,
    AI_EXTRACT(
        text => REVIEW,
        responseFormat => {
            'truck_name': 'What food truck or brand is mentioned?',
            'dish': 'What specific dish or menu item is mentioned?',
            'issue_type': 'What type of issue did the customer experience?'
        },
        scores => TRUE
    ) AS extraction_with_scores
FROM TB_VOC.ANALYTICS.TRUCK_REVIEWS_V_SAMPLE
LIMIT 5;

In [ ]:
%%sql -r dataframe_3
-- Batch extraction at scale: materialize results into a table
CREATE OR REPLACE TABLE TB_VOC.ANALYTICS.EXTRACTED_REVIEWS AS
SELECT
    TRUCK_BRAND_NAME,
    REVIEW,
    AI_EXTRACT(
        text => REVIEW,
        responseFormat => {
            'truck_name': 'What food truck or brand is mentioned?',
            'dish': 'What specific dish or menu item is mentioned?',
            'issue_type': 'What type of issue did the customer experience (food quality, service, wait time, cleanliness, none)?',
            'would_recommend': 'Would the customer recommend this food truck (yes, no, unclear)?'
        }
    ):response AS extracted
FROM TB_VOC.ANALYTICS.TRUCK_REVIEWS_V_SAMPLE;

In [ ]:
%%sql -r dataframe_4
-- Query the flattened extraction results
SELECT 
    TRUCK_BRAND_NAME,
    extracted:truck_name::VARCHAR AS truck_name,
    extracted:dish::VARCHAR AS dish,
    extracted:issue_type::VARCHAR AS issue_type,
    extracted:would_recommend::VARCHAR AS would_recommend
FROM TB_VOC.ANALYTICS.EXTRACTED_REVIEWS
WHERE extracted:issue_type::VARCHAR != 'none'
LIMIT 20;

## 2. Image Extraction with AI_EXTRACT

Extract structured data from food truck photos using AI_EXTRACT with file input.

In [ ]:
%%sql -r dataframe_5
-- Extract menu information from food truck images
SELECT
    IMAGE_PATH,
    AI_EXTRACT(
        file => TO_FILE('@TB_VOC.MEDIA.IMAGES', IMAGE_PATH),
        responseFormat => {
            'brand_name': 'What is the food truck or restaurant brand name visible?',
            'menu_items': 'What menu items or dishes are visible?',
            'prices': 'What prices are shown?',
            'location_clues': 'What location indicators are visible (street signs, landmarks)?'
        }
    ) AS extracted_data
FROM TB_VOC.MEDIA.IMAGE_TABLE
LIMIT 5;

In [ ]:
%%sql -r dataframe_6
-- Batch image extraction using directory table
SELECT
    RELATIVE_PATH,
    AI_EXTRACT(
        file => TO_FILE('@TB_VOC.MEDIA.IMAGES', RELATIVE_PATH),
        responseFormat => {
            'brand_name': 'What is the food truck brand name?',
            'menu_items': 'What menu items are visible?',
            'prices': 'What prices are shown?'
        }
    ) AS extracted_data
FROM DIRECTORY(@TB_VOC.MEDIA.IMAGES);

## 3. Video Analysis with AI_COMPLETE

Analyze social media video clips using AI_COMPLETE with gemini-3.1-pro and structured JSON output. Video support is in public preview.

In [ ]:
%%sql -r dataframe_7
-- Extract structured metadata from video clips
SELECT
    VIDEO_PATH,
    AI_COMPLETE(
        'gemini-3.1-pro',
        'Analyze this food truck social media video. Extract structured metadata.',
        TO_FILE('@TB_VOC.MEDIA.VIDEO', VIDEO_PATH),
        {},
        {
            'type': 'json',
            'schema': {
                'type': 'object',
                'properties': {
                    'sentiment': {'type': 'string'},
                    'summary': {'type': 'string'},
                    'brands_mentioned': {'type': 'array', 'items': {'type': 'string'}},
                    'dishes_shown': {'type': 'array', 'items': {'type': 'string'}},
                    'setting': {'type': 'string'},
                    'audience_type': {'type': 'string'}
                },
                'required': ['sentiment', 'summary', 'brands_mentioned', 'dishes_shown', 'setting', 'audience_type']
            }
        }
    ) AS video_metadata
FROM TB_VOC.MEDIA.VIDEO_TABLE
LIMIT 3;

In [ ]:
%%sql -r dataframe_8
-- Materialize video insights for downstream analytics
CREATE OR REPLACE TABLE TB_VOC.ANALYTICS.VIDEO_INSIGHTS AS
SELECT
    VIDEO_PATH,
    AI_COMPLETE(
        'gemini-3.1-pro',
        'Analyze this food truck social media video clip. Identify the brand, products shown, overall sentiment, and a brief summary of what is happening.',
        TO_FILE('@TB_VOC.MEDIA.VIDEO', VIDEO_PATH),
        {},
        {
            'type': 'json',
            'schema': {
                'type': 'object',
                'properties': {
                    'sentiment': {'type': 'string'},
                    'summary': {'type': 'string'},
                    'brands_mentioned': {'type': 'array', 'items': {'type': 'string'}},
                    'dishes_shown': {'type': 'array', 'items': {'type': 'string'}},
                    'setting': {'type': 'string'},
                    'audience_type': {'type': 'string'}
                },
                'required': ['sentiment', 'summary', 'brands_mentioned', 'dishes_shown', 'setting', 'audience_type']
            }
        }
    ) AS video_metadata
FROM TB_VOC.MEDIA.VIDEO_TABLE;

## 4. Audio Extraction

Transcribe voicemails with AI_TRANSCRIBE and extract structured fields from the transcriptions using AI_COMPLETE.

In [ ]:
%%sql -r dataframe_9
-- Transcribe audio files
SELECT
    AUDIO_PATH,
    AI_TRANSCRIBE(
        TO_FILE('@TB_VOC.MEDIA.AUDIO', AUDIO_PATH)
    ) AS transcription_result
FROM TB_VOC.MEDIA.AUDIO_TABLE
LIMIT 3;

In [ ]:
%%sql -r dataframe_10
-- Extract structured fields from transcriptions
SELECT
    AUDIO_PATH,
    AI_COMPLETE(
        'claude-sonnet-4-6',
        CONCAT(
            'Extract structured information from this customer voicemail transcription. ',
            'Return a JSON object with: caller_issue, truck_name, urgency (low/medium/high), and action_required. ',
            'Transcription: ',
            AI_TRANSCRIBE(TO_FILE('@TB_VOC.MEDIA.AUDIO', AUDIO_PATH)):text::VARCHAR
        ),
        {},
        {
            'type': 'json',
            'schema': {
                'type': 'object',
                'properties': {
                    'caller_issue': {'type': 'string'},
                    'truck_name': {'type': 'string'},
                    'urgency': {'type': 'string'},
                    'action_required': {'type': 'string'}
                },
                'required': ['caller_issue', 'truck_name', 'urgency', 'action_required']
            }
        }
    ) AS extracted_issue
FROM TB_VOC.MEDIA.AUDIO_TABLE
LIMIT 3;

## 5. Unified Batch Pipeline

Combine all modalities into a single extraction pipeline.

In [ ]:
%%sql -r dataframe_11
-- Create a unified extraction table across all modalities
CREATE OR REPLACE TABLE TB_VOC.ANALYTICS.UNIFIED_EXTRACTIONS AS

-- Text reviews
SELECT
    'text' AS modality,
    REVIEW AS source_content,
    NULL AS file_path,
    AI_EXTRACT(
        text => REVIEW,
        responseFormat => {
            'truck_name': 'What food truck is mentioned?',
            'issue_type': 'What issue did the customer experience?',
            'dish': 'What dish is mentioned?'
        }
    ):response AS extracted_data
FROM TRUCK_REVIEWS_V_SAMPLE

UNION ALL

-- Images
SELECT
    'image' AS modality,
    NULL AS source_content,
    RELATIVE_PATH AS file_path,
    AI_EXTRACT(
        file => TO_FILE('@TB_VOC.MEDIA.IMAGES', RELATIVE_PATH),
        responseFormat => {
            'brand_name': 'What is the food truck brand name?',
            'menu_items': 'What menu items are visible?',
            'prices': 'What prices are shown?'
        }
    ):response AS extracted_data
FROM DIRECTORY(@TB_VOC.MEDIA.IMAGES)

UNION ALL

-- Video
SELECT
    'video' AS modality,
    NULL AS source_content,
    VIDEO_PATH AS file_path,
    PARSE_JSON(
        REGEXP_REPLACE(
            AI_COMPLETE(
                'gemini-2.5-flash',
                'Extract brand names and dishes shown from this food truck video. Return ONLY a JSON object with keys: brand_name, dishes_shown, sentiment. No markdown, no code fences.',
                TO_FILE('@TB_VOC.MEDIA.VIDEO', VIDEO_PATH)
            ),
            '^```[a-z]*\\n?|\\n?```$', ''
        )
    ) AS extracted_data
FROM TB_VOC.MEDIA.VIDEO_TABLE

UNION ALL

-- Audio
SELECT
    'audio' AS modality,
    NULL AS source_content,
    AUDIO_PATH AS file_path,
    AI_COMPLETE(
        'claude-sonnet-4-6',
        CONCAT(
            'Extract structured information from this customer voicemail transcription. ',
            'Return a JSON object with: caller_issue, truck_name, urgency (low/medium/high), and action_required. ',
            'Transcription: ',
            AI_TRANSCRIBE(TO_FILE('@TB_VOC.MEDIA.AUDIO', AUDIO_PATH)):text::VARCHAR
        ),
        {},
        {
            'type': 'json',
            'schema': {
                'type': 'object',
                'properties': {
                    'caller_issue': {'type': 'string'},
                    'truck_name': {'type': 'string'},
                    'urgency': {'type': 'string'},
                    'action_required': {'type': 'string'}
                },
                'required': ['caller_issue', 'truck_name', 'urgency', 'action_required']
            }
        }
    ) AS extracted_data
FROM TB_VOC.MEDIA.AUDIO_TABLE;

In [ ]:
%%sql -r dataframe_21
SELECT
  *
FROM
  TB_VOC.ANALYTICS.UNIFIED_EXTRACTIONS
;

In [ ]:
%%sql -r dataframe_12
-- Query unified results across all modalities
SELECT
    modality,
    file_path,
    extracted_data:truck_name::VARCHAR AS truck_name,
    extracted_data:issue_type::VARCHAR AS issue_type,
    extracted_data:urgency::VARCHAR AS urgency
FROM TB_VOC.ANALYTICS.UNIFIED_EXTRACTIONS
WHERE extracted_data:urgency::VARCHAR = 'high'
   OR extracted_data:issue_type::VARCHAR NOT IN ('none', 'null');

## 6. AI Function Studio: Create

Create a reusable custom AI function that encapsulates the text extraction logic.

In [ ]:
%%sql -r dataframe_13
USE TB_VOC.ANALYTICS;

DROP FUNCTION IF EXISTS TB_VOC.ANALYTICS.EXTRACT_REVIEW_FIELDS(VARCHAR);

CALL SNOWFLAKE.CORTEX.CREATE_AI_FUNCTION(
    'TB_VOC.ANALYTICS.EXTRACT_REVIEW_FIELDS',
    'claude-sonnet-4-6',
    'You are a data extraction specialist for a food truck company. Extract structured information from customer reviews accurately and concisely. If a field cannot be determined from the review, return null for that field.',
    'Extract the following fields from this customer review:\n\nReview: {REVIEW_TEXT}\n\nExtract: truck_name, dish_mentioned, issue_type (food quality/service/wait time/cleanliness/none), would_recommend (yes/no/unclear)',
    PARSE_JSON('[
        {"name": "REVIEW_TEXT", "sql_type": "VARCHAR", "description": "The customer review text"}
    ]'),
    PARSE_JSON('[
        {"name": "truck_name", "json_type": "string", "description": "Food truck brand name"},
        {"name": "dish_mentioned", "json_type": "string", "description": "Specific dish or menu item"},
        {"name": "issue_type", "json_type": "string", "description": "Type of issue: food quality, service, wait time, cleanliness, or none"},
        {"name": "would_recommend", "json_type": "string", "description": "Would recommend: yes, no, or unclear"}
    ]')
);

In [ ]:
%%sql -r dataframe_14
-- Test the custom function
SELECT
    REVIEW,
    EXTRACT_REVIEW_FIELDS(REVIEW):truck_name::VARCHAR AS truck_name,
    EXTRACT_REVIEW_FIELDS(REVIEW):dish_mentioned::VARCHAR AS dish,
    EXTRACT_REVIEW_FIELDS(REVIEW):issue_type::VARCHAR AS issue_type,
    EXTRACT_REVIEW_FIELDS(REVIEW):would_recommend::VARCHAR AS recommendation
FROM TRUCK_REVIEWS_V
LIMIT 10;

## 7. AI Function Studio: Evaluate

Evaluate the custom function's accuracy against labeled test data.

In [ ]:
%%sql -r dataframe_15
-- Preview the labeled evaluation data
SELECT
    *
FROM
    TB_VOC.ANALYTICS.EXTRACTION_EVAL_DATA
LIMIT 5;

In [ ]:
%%sql -r dataframe_16
-- Evaluate the function
USE TB_VOC.ANALYTICS;

CALL SNOWFLAKE.CORTEX.EVALUATE_AI_FUNCTION(
    'TB_VOC.ANALYTICS.EXTRACT_REVIEW_FIELDS',
    'TB_VOC.ANALYTICS.EXTRACTION_EVAL_DATA',
    ARRAY_CONSTRUCT('REVIEW_TEXT'),
    'EXPECTED_OUTPUT',
    'llm_judge',
    'claude-sonnet-4-6',
    NULL,
    NULL,
    PARSE_JSON('{"task_description": "Evaluate whether the extracted fields (truck_name, dish_mentioned, issue_type, would_recommend) match the expected values. Focus on semantic equivalence rather than exact string matching."}'),
    500,
    NULL,
    NULL
);

## 8. AI Function Studio: Optimize

Optimize the function to improve extraction accuracy. The optimizer iteratively modifies the function body to find better-performing variants.

In [ ]:
%%sql -r dataframe_17
-- Optimize the extraction function
USE TB_VOC.ANALYTICS;

CALL SNOWFLAKE.CORTEX.OPTIMIZE_AI_FUNCTION(
    'TB_VOC.ANALYTICS.EXTRACT_REVIEW_FIELDS',
    'TB_VOC.ANALYTICS.EXTRACTION_EVAL_DATA',
    'EXPECTED_OUTPUT',
    ARRAY_CONSTRUCT('REVIEW_TEXT'),
    'llm_judge',
    ARRAY_CONSTRUCT('claude-sonnet-4-6'),
    'claude-sonnet-4-6',
    NULL,
    'demo',
    0.5,
    0.0,
    8192,
    PARSE_JSON('{"task_description": "Evaluate whether the extracted fields (truck_name, dish_mentioned, issue_type, would_recommend) match the expected values. Focus on semantic equivalence."}'),
    NULL,
    NULL,
    NULL,
    'body',
    'EXTRACT_REVIEW_FIELDS_OPT'
);

In [ ]:
%%sql -r dataframe_18
-- View optimization results
SHOW RUN METRICS IN EXPERIMENT TB_VOC.ANALYTICS.EXTRACT_REVIEW_FIELDS_OPT;

In [ ]:
%%sql -r dataframe_19
-- Test the optimized function
SELECT
    REVIEW,
    EXTRACT_REVIEW_FIELDS(REVIEW):truck_name::VARCHAR AS truck_name,
    EXTRACT_REVIEW_FIELDS(REVIEW):dish_mentioned::VARCHAR AS dish,
    EXTRACT_REVIEW_FIELDS(REVIEW):issue_type::VARCHAR AS issue_type,
    EXTRACT_REVIEW_FIELDS(REVIEW):would_recommend::VARCHAR AS recommendation
FROM TRUCK_REVIEWS_V
LIMIT 10;

## Conclusion

You have built a complete batch data extraction pipeline using Snowflake Cortex AI Functions:

- **AI_EXTRACT** for structured field extraction from text and images
- **AI_COMPLETE** with gemini-3.1-pro for video analysis with structured JSON output
- **AI_TRANSCRIBE** + **AI_COMPLETE** for audio transcription and extraction
- **AI Function Studio** to create, evaluate, and optimize a reusable extraction function

All processing stayed within Snowflake's secure environment with no data movement required.